In [ ]:
!pip install -q transformers datasets scikit-learn accelerate gradio

#### Imports and setup

In [ ]:
import random
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score,f1_score

#### We do the seed to get similar results everytime.

In [ ]:
SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

#### Load IMDB dataset from Huggingface with 25000 reviews

In [ ]:
dataset = load_dataset("imdb")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [ ]:
sample = dataset["train"][0]
print("Label:",sample["label"],"(0=negative, 1=positive)")
print("\nReview:", sample["text"][:500],"...")

Label: 0 (0=negative, 1=positive)

Review: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attent ...


#### take small subset for training

In [ ]:
TRAIN_SIZE = 2000
TEST_SIZE = 500

small_train = dataset["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))
small_test = dataset["test"].shuffle(seed=SEED).select(range(TEST_SIZE))

# split the training data in train+validation
split = small_train.train_test_split(test_size=0.1, seed=SEED)
train_ds = split["train"]
val_ds = split["test"]

print("Train size : ",len(small_train))
print("Validation size : ",len(val_ds))
print("Test size : ",len(small_test))

Train size :  2000
Validation size :  200
Test size :  500


#### Tokenize the text

In [ ]:
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 256
#download the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
  return tokenizer(batch["text"], max_length=MAX_LENGTH, truncation=True)

#tokenize all three sets
train_tokenize= train_ds.map(tokenize, batched=True)
val_tokenize = val_ds.map(tokenize, batched=True)
test_tokenize = small_test.map(tokenize, batched=True)

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

#### Load pretrained BERT for classification

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


#### metrics and data collator

In [ ]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)
  f1 = f1_score(labels, predictions)
  acc = accuracy_score(labels, predictions)
  return {"accuracy": acc, "f1": f1}

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

#### Setup Trainer

In [ ]:
training_args = TrainingArguments(
    output_dir="./bert_imdb_output",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8, #takes 8 reviews at a time
    per_device_eval_batch_size=8,
    num_train_epochs=2,           #increase performance but check if model is not overfitting
    weight_decay=0.01,
    fp16=torch.cuda.is_available(), #true if GPU available
    report_to="none",
    seed=SEED,
    logging_steps=50
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenize,
    eval_dataset=val_tokenize,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

#### Fine tuning the model

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.362839,0.304890,0.880000,0.869565
2,0.250800,0.486496,0.880000,0.878788


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=450, training_loss=0.3408034430609809, metrics={'train_runtime': 87.148, 'train_samples_per_second': 41.309, 'train_steps_per_second': 5.164, 'total_flos': 473271010828800.0, 'train_loss': 0.3408034430609809, 'epoch': 2.0})

#### evaluate on test set

In [ ]:
final_results = trainer.evaluate(eval_dataset=test_tokenize)
print("Final Results:")
for k,v in final_results.items():
  print(f"{k}: {v:.4f}")

Final Results:
eval_loss: 0.4214
eval_accuracy: 0.8860
eval_f1: 0.8885
eval_runtime: 3.1393
eval_samples_per_second: 159.2730
eval_steps_per_second: 20.0680
epoch: 2.0000


#### Predict new reviews

In [ ]:
id2label = {0: "NEGATIVE", 1: "POSITIVE"}

def predict_sentiment(text):
  inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt", max_length=MAX_LENGTH).to(model.device)

  model.eval()
  with torch.no_grad():
    logits = model(**inputs).logits

  probs = torch.softmax(logits, dim=1)[0]
  pred_id = torch.argmax(probs).item()

  return{
      "label":id2label[pred_id],
      "confidence_score": round(probs[pred_id].item(), 4),
  }

reviews = [
      "This movie waas absolutely fantastic. Brilliant acting and cinematography",
      "I hated this film. It was boring, too long and poorl written.",
      "It was Okay - some good moments but overall forgetable."
]

for r in reviews:
  print(predict_sentiment(r), "->" , r[:60], "...")


{'label': 'POSITIVE', 'confidence_score': 0.9935} -> This movie waas absolutely fantastic. Brilliant acting and c ...
{'label': 'NEGATIVE', 'confidence_score': 0.9912} -> I hated this film. It was boring, too long and poorl written ...
{'label': 'POSITIVE', 'confidence_score': 0.8797} -> It was Okay - some good moments but overall forgetable. ...


#### Saving the model and Tokenizer

In [ ]:
SAVE_DIR = "./imdb_bert"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Saved model and tokenizer:", SAVE_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model and tokenizer: ./imdb_bert


#### GRADIO UI

In [ ]:
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load the saved model and tokenizer
model_path = "./imdb_bert"
reload_model = AutoModelForSequenceClassification.from_pretrained(model_path)
reload_tokenizer = AutoTokenizer.from_pretrained(model_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
reload_model.to(device)
reload_model.eval()

print("Model and tokenizer loaded.", device)

# Inference function

id2label = {0: "NEGATIVE", 1: "POSITIVE"}
MAX_LENGTH=256

def predict_sentiment(text):
  inputs = reload_tokenizer(text, padding=True, truncation=True, return_tensors="pt", max_length=MAX_LENGTH).to(device)

  with torch.no_grad():
    logits = reload_model(**inputs).logits
  probs = torch.softmax(logits, dim=1)[0]
  pred_id = torch.argmax(probs).item()

  return id2label[pred_id], probs[pred_id].item()

# GRADIO UI

def classify_review(review):
  if not review.strip():
    return "Please enter a review"

  label, confidence = predict_sentiment(review)
  emoji ="🍅" if label =="POSITIVE" else "🤢"
  return f"{emoji} **{label.upper()}** (confidence: {confidence:.2f})"

demo = gr.Interface(
    fn = classify_review,
    inputs = gr.Textbox(
        lines=5,
        placeholder="Enter your review here...",
        label="Rotten Tomatos Review"
    ),
    outputs= gr.Markdown(label="Prediction"),
    title="🎬 Rotten Tomatoes “Splat” (Rotten Score)",
    description="Fine tuned BERT predicitng weather a movie is fresh or rotten",
    flagging_mode ="never",
    examples=[
        ["This movie waas absolutely fantastic. Brilliant acting and cinematography"],
        ["I hated this film. It was boring, too long and poorl written."],
        ["It was Okay - some good moments but overall forgetable."]
    ]
)

demo.launch(share=True)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model and tokenizer loaded. cuda
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0b0869ba47b3b8b4cf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
